# 06 · Ajuste de hiperparámetros y diagnóstico

**Fase CRISP-DM:** modelado y evaluación.

**Objetivo:** cerrar el experimento A. Para las mejores configuraciones del notebook 05:

1. ajustar los hiperparámetros con Optuna **dentro de cada partición** (ajuste anidado, I-17) y ver si cambia el resultado;
2. comprobar que la ventaja no es un efecto de haber elegido entre muchas configuraciones;
3. diagnosticar dónde falla el modelo elegido y qué variables usa;
4. concluir, por segmento, si supera al estacional ingenuo.

**Regla de privacidad:** solo usa la serie agregada por mes y segmento.

In [ ]:
import time
import warnings

import exp_a
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from lojagas.comun import lakehouse
from lojagas.ml.walk_forward import particiones

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="IProgress not found")
import shap  # noqa: E402  (después de silenciar el aviso de su barra de progreso)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)
exp_a.estilo_graficos()

H = exp_a.HORIZONTE
RAIZ = exp_a.raiz_lakehouse()
N_CANDIDATOS = 3  # configuraciones que pasan al ajuste
N_TRIALS = 20  # el mismo presupuesto de búsqueda para todas
PLIEGUES = 4  # meses de validación interna dentro de cada partición
MESES_CONFIRMACION = 7  # últimos meses de prueba, apartados para confirmar la elección

serie = exp_a.cargar_serie()
datos = exp_a.tabla_modelado(serie)
SEGMENTOS = sorted(datos["segmento"].unique())
COLUMNAS = [*SEGMENTOS, "macro", "TOTAL"]

pred = lakehouse.leer(RAIZ, "predicciones", exp_a.TABLA_PREDICCIONES)
pred["mes"] = pd.to_datetime(pred["mes"]).astype("datetime64[ns]")
pred = pred.sort_values([*exp_a.CLAVES, "mes", "segmento"]).reset_index(drop=True)
assert set(pred["mes"]) <= set(datos["mes"]), "las predicciones no corresponden a esta serie"

tabla_05 = exp_a.comparar(pred)
CANDIDATOS = [c for c in tabla_05.index if c[1] != "referencia"][:N_CANDIDATOS]
print("Candidatos (los de menor WAPE macro en el notebook 05):")
display(tabla_05.loc[CANDIDATOS, COLUMNAS].round(2))

## 1. Ajuste anidado

En cada partición, Optuna prueba hiperparámetros usando solo los meses de entrenamiento de esa partición: los últimos meses hacen de validación interna, con el mismo horizonte. El mes de prueba nunca participa en la búsqueda. Los valores del catálogo son siempre el primer intento.

In [ ]:
ajustes, elegidos = [], {}
for nombre, modo, escala in CANDIDATOS:
    inicio = time.time()
    p, e = exp_a.ajustar_anidado(
        nombre, datos, modo=modo, escala=escala, n_trials=N_TRIALS, pliegues=PLIEGUES
    )
    ajustes.append(p)
    elegidos[(nombre, modo, escala)] = e
    print(f"{nombre} · {modo} · {escala}: {len(e)} estudios en {time.time() - inicio:.0f} s")

ajustado = pd.concat(ajustes, ignore_index=True)
assert ajustado["pred"].notna().all()
version = lakehouse.escribir(ajustado, RAIZ, "predicciones", exp_a.TABLA_AJUSTE, modo="overwrite")
todo = pd.concat([pred, ajustado], ignore_index=True)
tabla = exp_a.comparar(todo)

In [ ]:
filas = []
for nombre, modo, escala in CANDIDATOS:
    antes = tabla.loc[(nombre, modo, escala)]
    despues = tabla.loc[(f"{nombre}+ajuste", modo, escala)]
    interno = elegidos[(nombre, modo, escala)]["wape_interno"].mean()
    filas.append(
        {
            "configuracion": f"{nombre} · {modo} · {escala}",
            "macro_catalogo": antes["macro"],
            "macro_ajustado": despues["macro"],
            "cambio_macro_%": (despues["macro"] / antes["macro"] - 1) * 100,
            "TOTAL_catalogo": antes["TOTAL"],
            "TOTAL_ajustado": despues["TOTAL"],
            "wape_validacion_interna": interno,
        }
    )
print("¿Mejora el ajuste? (cambio negativo = menos error)")
display(pd.DataFrame(filas).set_index("configuracion").round(2))

Si la validación interna promete mucho menos error del que luego se obtiene en la prueba, la búsqueda está sobreajustando a muy pocos meses. La tabla siguiente muestra cuánto cambian los hiperparámetros elegidos de una partición a otra: si saltan de un extremo al otro, los datos no alcanzan para elegirlos.

In [ ]:
for config, e in elegidos.items():
    print(" · ".join(config))
    numericos = e.drop(columns=["fold", "wape_interno"]).select_dtypes("number").columns
    display(e.groupby("grupo")[list(numericos)].agg(["median", "min", "max"]).round(3))

## 2. Configuración elegida y comprobación de la elección

Se elige la de menor WAPE macro entre las candidatas, con y sin ajuste. Elegir la mejor de muchas configuraciones sobre los mismos meses favorece a la que tuvo suerte. Para medirlo, los meses de prueba se parten en dos: con los primeros se elige y con los últimos, que no intervinieron en la elección, se confirma.

In [ ]:
modelos = tabla[tabla.index.get_level_values("modo") != "referencia"]
FINAL = modelos.index[0]
print("Configuración elegida:", " · ".join(FINAL))

meses = np.sort(todo["mes"].unique())
corte = meses[len(meses) - MESES_CONFIRMACION]
seleccion = exp_a.comparar(todo[todo["mes"] < corte])
confirmacion = exp_a.comparar(todo[todo["mes"] >= corte])
elegida_antes = seleccion[seleccion.index.get_level_values("modo") != "referencia"].index[0]
print(
    f"Con los primeros {len(meses) - MESES_CONFIRMACION} meses se habría elegido: "
    + " · ".join(elegida_antes)
)
filas = {}
comparadas = {"elegida con todo": FINAL, "elegida solo con la selección": elegida_antes}
for etiqueta, config in comparadas.items():
    for periodo, t in (("selección", seleccion), ("confirmación", confirmacion)):
        filas[(etiqueta, periodo)] = t.loc[config, [f"mejora_{c}" for c in COLUMNAS]]
print("Mejora sobre el estacional ingenuo (%) en cada período")
display(pd.DataFrame(filas).T.rename(columns=lambda c: c.removeprefix("mejora_")).round(1))

### Elección por segmento

El notebook 05 mostró que ninguna configuración gana en todos los segmentos. Aquí cada segmento elige su mejor método con los meses de selección, y las referencias sin aprendizaje también compiten: si ningún modelo las supera, el segmento se queda con una de ellas. La columna que cuenta es la de confirmación, medida en meses que no intervinieron en la elección.

In [ ]:
AJUSTADA = ("estacional_dias", "referencia", "toneladas")
COMPITEN = [c for c in seleccion.index if c[0] not in ("ingenuo", "media_12")]
filas, partes = [], []
for segmento in SEGMENTOS:
    elegida = seleccion.loc[COMPITEN, segmento].idxmin()
    filas.append(
        {
            "segmento": segmento,
            "elegida_en_seleccion": " · ".join(elegida),
            "wape_seleccion": seleccion.loc[elegida, segmento],
            "wape_confirmacion": confirmacion.loc[elegida, segmento],
            "estacional_en_confirmacion": confirmacion.loc[exp_a.REFERENCIA, segmento],
            "estacional_dias_en_confirmacion": confirmacion.loc[AJUSTADA, segmento],
            "mejora_en_confirmacion_%": confirmacion.loc[elegida, f"mejora_{segmento}"],
        }
    )
    filtro = (todo[exp_a.CLAVES] == pd.Series(dict(zip(exp_a.CLAVES, elegida, strict=True)))).all(
        axis=1
    )
    partes.append(todo[filtro & (todo["segmento"] == segmento)])
por_segmento = pd.DataFrame(filas).set_index("segmento")
display(por_segmento.round(2))

# La combinación: cada segmento con su método elegido, sumados para el total
combinada = pd.concat(partes, ignore_index=True).assign(
    modelo="combinada por segmento", modo="mixto", escala="mixta"
)
con_combinada = pd.concat([todo[todo["modo"] == "referencia"], combinada], ignore_index=True)
resumen_combinada = exp_a.comparar(con_combinada[con_combinada["mes"] >= corte])
print("Meses de confirmación: WAPE (%) de la combinación frente a las referencias")
display(resumen_combinada[COLUMNAS].round(2))

## 3. Diagnóstico de la configuración elegida

Se diagnostica una sola familia de modelo, la de menor WAPE macro, en los tres segmentos: muestra dónde funciona y dónde no.

In [ ]:
res = exp_a.resumir(todo).set_index([*exp_a.CLAVES, "segmento"]).sort_index()
metricas_cols = ["wape", "mape", "mae", "rmse", "r2", "sesgo_pct", "n"]
diagnostico = pd.concat(
    {"estacional ingenuo": res.loc[exp_a.REFERENCIA], "modelo elegido": res.loc[FINAL]}
)[metricas_cols]
display(diagnostico.round(2))

print("Prueba pareada del error absoluto mensual frente al estacional ingenuo")
pareada = exp_a.prueba_pareada(todo, FINAL)
display(pareada.round(3))

In [ ]:
def serie_de(config):
    filtro = (todo[exp_a.CLAVES] == pd.Series(dict(zip(exp_a.CLAVES, config, strict=True)))).all(
        axis=1
    )
    return todo[filtro]


final, referencia = serie_de(FINAL), serie_de(exp_a.REFERENCIA)
fig, ejes = plt.subplots(len(SEGMENTOS), 1, figsize=(10, 2.6 * len(SEGMENTOS)), sharex=True)
for eje, segmento in zip(ejes, SEGMENTOS, strict=True):
    f = final[final["segmento"] == segmento].sort_values("mes")
    r = referencia[referencia["segmento"] == segmento].sort_values("mes")
    eje.plot(f["mes"], f["y"], color=exp_a.TINTA, label="observado")
    eje.plot(r["mes"], r["pred"], color=exp_a.NARANJA, label="estacional ingenuo")
    eje.plot(f["mes"], f["pred"], color=exp_a.AZUL, label="modelo elegido")
    eje.set_title(f"{segmento} · toneladas en los meses de prueba")
    eje.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ejes[0].legend(loc="best", ncols=3)
plt.tight_layout()
plt.show()

In [ ]:
print("Error absoluto medio (%) por mes del año: modelo elegido menos estacional ingenuo")
print("(negativo = el modelo se equivoca menos; cada celda sale de una o dos observaciones)")
ape = pd.concat({"modelo": final, "referencia": referencia}, names=["quien"]).reset_index("quien")
ape["ape"] = (ape["pred"] - ape["y"]).abs() / ape["y"] * 100
por_mes = ape.assign(mes_del_año=ape["mes"].dt.month).pivot_table(
    index="mes_del_año", columns=["quien", "segmento"], values="ape"
)
display((por_mes["modelo"] - por_mes["referencia"]).round(1))

## 4. ¿Qué variables aportan?

La misma familia de modelo (con los hiperparámetros del catálogo) entrenada con distintos conjuntos de variables. Dice qué partes de la configuración A sostienen el resultado.

In [ ]:
familia, modo, escala = FINAL[0].removesuffix("+ajuste"), FINAL[1], FINAL[2]
base = exp_a.variables(escala)
cortos = [v for v in base if v.startswith(("lag_", "media_3")) and not v.startswith("lag_12")]
LAGS_6 = tuple(sorted({*exp_a.LAGS, 6}))
datos_6 = exp_a.tabla_modelado(serie, lags=LAGS_6)
assert len(datos_6) == len(datos)
VARIANTES = {
    "variables A": (datos, base),
    "sin días de venta": (datos, [v for v in base if not v.startswith("dias_")]),
    "sin segundo armónico": (datos, [v for v in base if v not in ("mes_sin2", "mes_cos2")]),
    "sin rezagos cortos": (datos, [v for v in base if v not in cortos]),
    "solo calendario": (datos, [v for v in base if v.startswith(("mes_", "dias_"))]),
    "+ rezago 6": (datos_6, exp_a.variables(escala, lags=LAGS_6)),
}
if "clientes_activos" in serie.columns:
    VARIANTES["+ clientes activos"] = (datos, exp_a.variables(escala, con_clientes=True))

partes = [todo[todo["modo"] == "referencia"]]
for etiqueta, (tabla_variante, features) in VARIANTES.items():
    p = exp_a.evaluar(familia, tabla_variante, modo=modo, escala=escala, features=features)
    partes.append(p.assign(modelo=etiqueta))
ablacion = exp_a.comparar(pd.concat(partes, ignore_index=True))
print(
    f"{familia} · {modo} · {escala} con distintas variables: WAPE (%) y mejora sobre el estacional"
)
display(
    ablacion[[*COLUMNAS, "mejora_macro", "mejora_TOTAL"]].droplevel(["modo", "escala"]).round(2)
)

### Importancia de las variables (SHAP)

Peso de cada variable en las predicciones del modelo de la última partición, como % del total. Con un modelo por segmento hay una columna por segmento.

In [ ]:
def fabrica_final(grupo):
    """El modelo de la última partición: con los hiperparámetros ajustados, si los tiene."""
    if not FINAL[0].endswith("+ajuste"):
        return exp_a.crear_modelo(familia, H)
    e = elegidos[(familia, modo, escala)]
    ultimo = e[(e["fold"] == e["fold"].max()) & (e["grupo"] == grupo)].iloc[0]
    return exp_a.modelo_con(familia, ultimo.drop(["fold", "grupo", "wape_interno"]).to_dict())


X, objetivo, _ = exp_a.matrices(datos, base, modo, escala)
tr, _ = list(particiones(datos["mes"], exp_a.MIN_ENTRENAMIENTO, horizonte=H))[-1]
pesos = {}
for grupo, g_tr in exp_a.grupos(datos["segmento"].to_numpy(), tr, modo):
    modelo = fabrica_final(grupo)
    modelo.fit(X.iloc[g_tr], objetivo.iloc[g_tr])

    def predecir(matriz, modelo=modelo):
        return modelo.predict(pd.DataFrame(matriz, columns=X.columns))

    valores = shap.Explainer(predecir, X.iloc[g_tr])(X.iloc[g_tr], silent=True)
    media = np.abs(valores.values).mean(axis=0)
    pesos[grupo] = pd.Series(media / media.sum() * 100, index=X.columns)
importancia = pd.DataFrame(pesos)
display(importancia.sort_values(importancia.columns[0], ascending=False).round(1))

## 5. Conclusión del experimento A

In [ ]:
conclusion = pd.DataFrame(
    {
        "wape_estacional_ingenuo": tabla.loc[exp_a.REFERENCIA, COLUMNAS],
        "wape_estacional_dias": tabla.loc[AJUSTADA, COLUMNAS],
        "wape_modelo": tabla.loc[FINAL, COLUMNAS],
        "mejora_%": tabla.loc[FINAL, [f"mejora_{c}" for c in COLUMNAS]].to_numpy(),
    }
)
conclusion["mejora_%_en_confirmacion"] = confirmacion.loc[
    FINAL, [f"mejora_{c}" for c in COLUMNAS]
].to_numpy()
conclusion = conclusion.join(pareada[["meses_en_que_gana", "meses", "p_wilcoxon"]])
print("Modelo elegido:", " · ".join(FINAL))
display(conclusion.round(3))

In [ ]:
_ = exp_a.registrar_mlflow(
    tabla.loc[[c for c in tabla.index if c[0].endswith("+ajuste") or c[1] == "referencia"]],
    "06_ajuste_anidado",
    {
        "horizonte": H,
        "min_entrenamiento": exp_a.MIN_ENTRENAMIENTO,
        "n_trials": N_TRIALS,
        "pliegues_internos": PLIEGUES,
        "elegida": " · ".join(FINAL),
        "delta_version_ajuste": version,
    },
)

## Decisiones

Las cifras quedan en las salidas de este notebook, que no se suben al repositorio.

1. **La configuración A supera al estacional ingenuo en el doméstico y en el total**, con una diferencia significativa en la prueba pareada y confirmada en los meses apartados. **No lo supera en el agroindustrial ni en el industrial.**
2. **El método se adopta por segmento, y con una regla exigente.** Un modelo reemplaza al estacional en un segmento solo si lo supera de forma significativa; si no, se usa el estacional ajustado por días de venta. Elegir simplemente la configuración de menor error en los meses de selección falló en los dos segmentos chicos: lo que ganó ahí perdió en la confirmación.
3. **El ajuste anidado de hiperparámetros no mejora el resultado.** Con cuatro meses de validación interna, los valores elegidos saltan de un extremo al otro del rango entre particiones y la validación interna promete menos error del que se obtiene. Se dejan los hiperparámetros conservadores del catálogo. Es un hallazgo para documentar (I-17), no un fallo del procedimiento.
4. **Lo que sostiene el resultado es el calendario:** los días de venta del mes, el nivel del último año cerrado y un perfil estacional fijo. Los rezagos cortos y los clientes activos no aportan en el doméstico. Un modelo que usa solo calendario rinde igual que el completo: es el candidato más simple y explicable.
5. **Para la configuración B, el margen está en el agroindustrial y el industrial,** donde el histórico no alcanza, y en los feriados para el doméstico.
6. **Límites.** Son menos de dos años de meses de prueba y una sola empresa. Las variables se diseñaron viendo la serie completa, aunque los pesos de los días de la semana se estimaron solo con el primer año.